# 第4日 ノートブック03：学習したモデルを評価する

# Day 4 Notebook 03: Evaluating the Trained Model

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 学習に使っていない画像で性能を測れる | 1. Evaluate on images never used for training |
| 2. precision と recall の意味を説明できる | 2. Explain precision and recall |
| 3. 混同行列を読める | 3. Read a confusion matrix |
| 4. 成功例と失敗例を記録できる | 4. Record successful and failed predictions |
| 5. データセットの改善案を提案できる | 5. Propose dataset improvements |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 140 分 / 140 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第4日ノートブック02 で学習を完了し、`weights/best.pt` があること。 / You finished the training in Day 4 Notebook 02 and `weights/best.pt` exists. |
| 必要なライブラリ / Required libraries | torch, ultralytics, opencv-python, numpy, matplotlib, pandas |
| 使用カーネル / Jupyter kernel | aivision |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 評価指標（precision / recall / mAP） | Metrics (precision / recall / mAP) |
| 失敗例の画像 | Failure-case images |
| 改善提案 | Improvement proposals |

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師：評価の考え方 | Instructor: how to evaluate |
| 09:00–10:30 | Part 1〜Part 3（指標・評価・混同行列） | Part 1-Part 3 (metrics, evaluation, confusion matrix) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4〜Part 5（失敗例・改善） | Part 4-Part 5 (failures, improvements) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：結果の確認と午後の説明 | Instructor: results review and briefing |
| 13:50–16:50 | 演習・提出物 | Exercises and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("cv2", "opencv-python"),
    ("torch", "torch"),
    ("ultralytics", "ultralytics"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

In [ ]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

#### 日本語：学習した重みを探す
**日本語：** 学習したモデルは `training_results/day04/.../weights/best.pt` にあります。
見つからない場合は、学習（04_02）を先に実行してください。

**日本語：** **学習していないモデルで評価することはできません。** 数字を作ってはいけません。

**English:** The trained model is at `training_results/day04/.../weights/best.pt`. If it is missing,
run the training notebook (04_02) first.

**You cannot evaluate a model you never trained, and you must never invent the numbers.**
#### English: Find the trained weights

In [ ]:
runs_root = COURSE_ROOT / "training_results" / "day04"
candidates = sorted(runs_root.glob("*/weights/best.pt")) if runs_root.is_dir() else []

print("見つかった重み / weights found:", len(candidates))
for path in candidates:
    print("  ", path.relative_to(COURSE_ROOT), f"({path.stat().st_size} bytes)")

if not candidates:
    print()
    print("学習がまだです。/ No trained weights yet.")
    print("04_02 の学習セルを RUN_TRAINING = True で実行してください。")
    print("Run the training cell in 04_02 with RUN_TRAINING = True.")

In [ ]:
from ultralytics import YOLO

if not candidates:
    raise FileNotFoundError(
        "学習済みの重みが見つかりません。04_02 を先に実行してください。"
        " / no trained weights found; run 04_02 first."
    )

best_weights = candidates[-1]          # 最新の学習結果 / the most recent run
model = YOLO(str(best_weights))
print("読み込みました / loaded:", best_weights.relative_to(COURSE_ROOT))
print("クラス / classes:", model.names)

---

## Part 1　評価の指標 / Evaluation metrics

#### 日本語：precision と recall
**日本語：** 2 つの指標は、**別のことを測って** います。

```text
precision（適合率） = 正しく検出した数 ÷ 検出した総数
                     -> 「検出したものは、どれくらい本物か」

recall（再現率）    = 正しく検出した数 ÷ 実際の欠陥の総数
                     -> 「本物の欠陥を、どれくらい捕まえられたか」
```

| 指標 | 低いと何が起きるか |
|---|---|
| precision が低い | **誤報が多い**。現場が疲れる、ラインが止まる |
| recall が低い | **見逃しが多い**。不良品を出荷する |

**日本語：** 両方を同時に上げるのは難しく、**トレードオフ** があります。
`confidence` のしきい値を動かすと、片方が上がり片方が下がります。

**English:** The two metrics measure **different things**.

```text
precision = true detections / all detections
          -> "of what we flagged, how much was real?"

recall    = true detections / all real defects
          -> "of all real defects, how many did we catch?"
```

| Metric | What happens when it is low |
|---|---|
| low precision | **many false alarms**: operators tire, the line stops |
| low recall | **many misses**: defective products ship |

Improving both at once is hard; there is a **trade-off**. Moving the confidence threshold raises one
and lowers the other.
#### English: Precision and recall

#### 日本語：mAP とは
**日本語：** **mAP**（mean Average Precision）は、precision と recall を
しきい値全体にわたってまとめた指標です。YOLO の学習ログで `mAP50` と `mAP50-95` が出てきます。

- `mAP50` … IoU 0.5 以上を「当たり」としたときの総合点
- `mAP50-95` … IoU 0.5〜0.95 を細かく変えて平均した、より厳しい総合点

**日本語：** 製造の現場では、**mAP より precision / recall のほうが理解しやすい** ことが多いです。
「見逃しが何 % か」のほうが、現場で意味を持ちます。

**English:** **mAP** (mean Average Precision) summarises precision and recall across all thresholds.
The YOLO training log reports `mAP50` and `mAP50-95`.

- `mAP50`: overall score counting IoU >= 0.5 as correct
- `mAP50-95`: a stricter score averaged over IoU 0.5 to 0.95

On a shop floor, **precision and recall are usually easier to discuss** than mAP: "what percentage
do we miss?" carries meaning.
#### English: What mAP is

---

## Part 2　試験データで評価する / Evaluating on the test split

#### 日本語：`split="test"` を使う
**日本語：** `model.val()` は、指定した分割で評価します。
**必ず `split="test"`** を使ってください（学習に使っていないデータ）。

`split="train"` で測ると、丸暗記の成績が出て、実力を誤解します。

**English:** `model.val()` evaluates on the split you name. **Always use `split="test"`**, the data
never used for training.

Measuring on `split="train"` reports memorisation and misleads you about real skill.
#### English: Use `split="test"`

In [ ]:
metrics = model.val(
    data=(ASSETS / "yolo_dataset" / "data.yaml").as_posix(),
    split="test",
    imgsz=320,
    batch=8,
    workers=0,
    device="cpu",
    plots=True,
    verbose=False,
    project=str(COURSE_ROOT / "training_results" / "day04"),
    name="test_evaluation",
    exist_ok=True,
)

print("クラスごとの正解数 / ground-truth instances per class:")
for class_id, count in enumerate(metrics.nt_per_class):
    print(f"  {metrics.names[class_id]:<14} {int(count)}")
print()
print(f"precision : {metrics.box.mp:.3f}")
print(f"recall    : {metrics.box.mr:.3f}")
print(f"mAP50     : {metrics.box.map50:.3f}")
print(f"mAP50-95  : {metrics.box.map:.3f}")
print()
print("結果の保存先 / results saved to:", metrics.save_dir)

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
クラスごとの正解数 / ground-truth instances per class:
  black_spot     5
  scratch        7
  streak         6

precision : 0.872
recall    : 0.889
mAP50     : 0.918
mAP50-95  : 0.609

結果の保存先 / results saved to: .../training_results/day04/test_evaluation
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
ground-truth instances per class:
  black_spot     5
  scratch        7
  streak         6

precision : 0.872
recall    : 0.889
mAP50     : 0.918
mAP50-95  : 0.609
```

#### 日本語：数字をどう読むか
**日本語：** 64 枚の学習画像で学習したモデルです。数字は **高くないかもしれません**。
それは **正常な結果** です。少ないデータでは、実力は限られます。

| 見方 | 判断 |
|---|---|
| recall が低い | 見逃しが多い。データを増やす、しきい値を下げる |
| precision が低い | 誤報が多い。背景画像を増やす、しきい値を上げる |
| 両方とも低い | 学習が足りない、ラベルが間違っている |

**日本語：** **「良い数字が出るまで学習を繰り返す」のは間違い** です。
数字が悪い原因を、データとラベルに探してください。

**English:** The model saw 64 training images, so the numbers may not be high. **That is a normal
result**: with little data, real skill is limited.

| Reading | Interpretation |
|---|---|
| low recall | many misses: add data, lower the threshold |
| low precision | many false alarms: add background images, raise the threshold |
| both low | not enough training, or wrong labels |

**Retraining until the number looks good is the wrong approach.** Look for the cause in the data and
the labels.
#### English: How to read these numbers

---

## Part 3　混同行列 / Confusion matrix

#### 日本語：混同行列の読み方
**日本語：** 混同行列は、**「何を、何と間違えたか」** を 1 枚の表にしたものです。

行が **正解**、列が **予測** です。

|  | 予測: black_spot | 予測: scratch | 予測: background |
|---|---|---|---|
| 正解: black_spot | 正しく検出 | 傷と間違えた | **見逃し** |
| 正解: scratch | 黒点と間違えた | 正しく検出 | **見逃し** |
| 正解: background | **誤検出** | **誤検出** | 正しく「無し」 |

**日本語：** 対角線（左上から右下）が大きいほど良い結果です。
**`background` の列** に大きな数字があれば **見逃し** が多く、
**`background` の行** に大きな数字があれば **誤検出** が多い、と読めます。

**English:** A confusion matrix puts **what was confused with what** into one table: rows are the
truth, columns are the prediction.

|  | predicted black_spot | predicted scratch | predicted background |
|---|---|---|---|
| truth black_spot | correct | called it a scratch | **missed** |
| truth scratch | called it a spot | correct | **missed** |
| truth background | **false alarm** | **false alarm** | correctly "nothing" |

The larger the diagonal, the better. A large `background` **column** means many misses; a large
`background` **row** means many false alarms.
#### English: How to read a confusion matrix

In [ ]:
if (best_weights.parent.parent).is_dir():
    run_dir = best_weights.parent.parent
    print("学習の出力フォルダ / training run folder:", run_dir.relative_to(COURSE_ROOT))
    print()

    evaluation_dirs = sorted(
        (COURSE_ROOT / "training_results" / "day04").glob("test_evaluation*"),
        key=lambda folder_: folder_.stat().st_mtime,
    )
    candidates_png = []
    for folder in reversed(evaluation_dirs):
        candidates_png = sorted(folder.glob("confusion_matrix*.png"))
        if candidates_png:
            print("評価フォルダ / evaluation folder:", folder.relative_to(COURSE_ROOT))
            break

    candidates_png = candidates_png or sorted(run_dir.glob("confusion_matrix*.png"))
    for candidate in candidates_png:
        print("  ", candidate.name)

    if candidates_png:
        show([Image.open(p) for p in candidates_png[:2]],
             [p.name for p in candidates_png[:2]], figsize=(12, 5))
    else:
        print("混同行列の画像がありません。`plots=True` で評価したか確認してください。")
        print("No confusion matrix image; check that the evaluation ran with `plots=True`.")
else:
    print("学習の出力フォルダがありません / the training output folder is missing")

---

## Part 4　成功例と失敗例 / Successes and failures

#### 日本語：自分の目で確認する
**日本語：** 数字だけでは、現場で使えるか分かりません。**画像を見ます。**

下のセルは、試験データの画像に予測を描きます。
**緑** が予測、**赤** が正解（ラベル）です。ずれている画像を探してください。

**English:** Numbers alone do not tell you whether it works on the line. **Look at the images.**

The next cell draws predictions on the test images: **green** is the prediction, **red** is the
ground truth. Look for images where they disagree.
#### English: Check with your own eyes

In [ ]:
import random

dataset_dir = ASSETS / "yolo_dataset"
test_images = sorted((dataset_dir / "images" / "test").glob("*.png"))
random.seed(1)
chosen = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, path in zip(axes.ravel(), chosen):
    img = cv2.imread(str(path))
    height, width = img.shape[:2]

    # 正解を赤で描く / draw the ground truth in red
    label_path = dataset_dir / "labels" / "test" / f"{path.stem}.txt"
    truth_count = 0
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        class_id, xc, yc, w, h = [float(v) for v in line.split()]
        x1, y1 = int((xc - w / 2) * width), int((yc - h / 2) * height)
        x2, y2 = int((xc + w / 2) * width), int((yc + h / 2) * height)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 1)
        truth_count += 1

    prediction = model.predict(source=str(path), conf=0.25, imgsz=320, verbose=False)[0]
    pred_count = len(prediction.boxes)
    for box in prediction.boxes:
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 1)

    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{path.stem}\ntruth={truth_count} pred={pred_count}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("red = truth, green = prediction / 赤=正解, 緑=予測")
plt.tight_layout()
plt.show()

#### 日本語：失敗の型
**日本語：** 観察すると、失敗には型があります。

| 型 | 症状 | 原因の候補 |
|---|---|---|
| 見逃し | 赤い枠だけがある | 欠陥が薄い、小さすぎる、学習例が少ない |
| 誤検出 | 緑の枠だけがある | 背景画像が少ない、模様を欠陥と誤認 |
| 位置ずれ | 枠が半分ずれる | ラベルの付け方が不安定、しきい値が甘い |
| クラス誤り | 傷を黒点と判定 | 見た目が似ている、学習例の偏り |

**English:** Failures fall into recognizable types.

| Type | Symptom | Likely cause |
|---|---|---|
| miss | only a red box | faint or too small, too few training examples |
| false alarm | only a green box | too few background images, pattern mistaken for a defect |
| misaligned box | boxes half overlap | inconsistent labelling, loose threshold |
| wrong class | scratch called a spot | similar appearance, imbalanced examples |
#### English: Types of failure

#### 日本語：過学習（overfitting）
**日本語：** 学習用では high、試験用では low になる状態を **過学習** と呼びます。

- 原因：データが少ない、モデルが大きすぎる、学習しすぎ
- 対策：データを増やす、データを増やす、データを増やす（最も効く）

**日本語：** この研修の 64 枚は、過学習が起きる条件そのものです。
**良い数字が出なかったとしても、間違いではありません。**

**English:** High performance on training data and low on test data is **overfitting**.

- causes: too little data, too large a model, too many epochs
- fix: more data (the most effective), then more data, then more data

64 images is exactly the setting where overfitting occurs. **A modest score is not a mistake.**
#### English: Overfitting

---

## Part 5　第5日と共通の画像で記録する / Recording results for Day 5

#### 日本語：なぜ別の画像でも記録するのか
**日本語：** 第 5 日では、**YOLO と PatchCore を同じ画像で比べます**。

そのために、ここで **比較用の画像セット** に対する YOLO の結果を保存します。
対象は `assets/anomaly_dataset/pvc_sheet/test/` です。この画像は **学習に使っていません**。

**日本語：** 保存した JSON を第 5 日ノートブック 03 が読み込みます。

**English:** Day 5 compares **YOLO and PatchCore on the same images**.

So here we save YOLO's results on a **shared comparison set**:
`assets/anomaly_dataset/pvc_sheet/test/`. These images were **never used for training**.

Day 5 Notebook 03 reads the saved JSON.
#### English: Why record results on another image set

In [ ]:
shared_root = ASSETS / "anomaly_dataset" / "pvc_sheet" / "test"
shared_items = []

if not shared_root.is_dir():
    print("共通画像セットがありません / shared set missing:", shared_root)
    print("python assets\\scripts\\prepare_datasets.py --only anomaly を実行してください。")
else:
    for path in sorted(shared_root.rglob("*.png")):
        label = "good" if path.parent.name == "good" else "bad"
        prediction = model.predict(source=str(path), conf=0.25, imgsz=320, verbose=False)[0]
        confidences = [float(v) for v in prediction.boxes.conf.tolist()]
        shared_items.append({
            "image": path.relative_to(shared_root).as_posix(),
            "true_label": label,
            "yolo_detected": len(prediction.boxes) > 0,
            "yolo_detections": len(prediction.boxes),
            "yolo_max_conf": round(max(confidences), 4) if confidences else 0.0,
            "yolo_classes": sorted({prediction.names[int(c)] for c in prediction.boxes.cls.tolist()}),
        })

    print("処理した画像 / images processed:", len(shared_items))

In [ ]:
if shared_items:
    import json

    submission_dir = COURSE_ROOT / "submissions" / "day04"
    submission_dir.mkdir(parents=True, exist_ok=True)
    shared_json = submission_dir / "yolo_predictions_on_shared_set.json"

    payload = {
        "note": (
            "YOLO predictions on the shared comparison set "
            "(assets/anomaly_dataset/pvc_sheet/test). These images were not used for training."
        ),
        "weights": str(best_weights.relative_to(COURSE_ROOT)),
        "conf_threshold": 0.25,
        "images": shared_items,
    }
    shared_json.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    import pandas as pd
    frame = pd.DataFrame(shared_items)
    print(frame.groupby("true_label")["yolo_detected"].agg(["count", "sum"]).to_string())
    print()
    print("保存 / saved:", shared_json)
else:
    print("保存するデータがありません / nothing to save")

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：precision と recall を記録する
**日本語：** Part 2 の結果を、次の表に記入してください。

| 指標 | 値 | 現場での意味 |
|---|---|---|
| precision | | |
| recall | | |
| mAP50 | | |

**English:** Fill in the table from Part 2 and state what each number means on the line.
#### English: Exercise 1 (basic): record precision and recall
Done when all three values are recorded and both metrics are explained in your own words.

In [ ]:
# 演習 1 / Exercise 1
record = {
    "precision": None,
    "recall": None,
    "map50": None,
    "precision_meaning": "",
    "recall_meaning": "",
}
for key, value in record.items():
    print(f"{key:<20} {value if value not in (None, '') else '(未記入 / not written)'}")

#### 日本語：演習 2（中級）：失敗例を 2 枚保存する
**日本語：** Part 4 の画像から、**見逃し** と **誤検出** をそれぞれ 1 枚ずつ選び、
`submissions/day04/` に保存してください。ファイル名に `miss_` / `false_alarm_` を付けます。

**English:** From Part 4, pick one **miss** and one **false alarm**, and save them into
`submissions/day04/` with file names starting `miss_` and `false_alarm_`.
#### English: Exercise 2 (intermediate): save two failure images
Done when two files exist and you can describe the cause of each.

In [ ]:
# 演習 2 / Exercise 2
submission_dir = COURSE_ROOT / "submissions" / "day04"
submission_dir.mkdir(parents=True, exist_ok=True)

# 例 / example:
# image = cv2.imread(str(path))
# cv2.imwrite(str(submission_dir / "miss_001.png"), image)
print("保存先 / folder:", submission_dir)

#### 日本語：演習 3（発展）：改善提案を書く
**日本語：** 失敗の型を踏まえ、**次に何をするか** を優先順に 3 つ書いてください。
それぞれ「なぜそれが効くのか」も書いてください。

| 優先 | やること | なぜ効くか |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |

**English:** Based on the failure types, write three actions in priority order, each with a reason
why it helps.
#### English: Exercise 3 (challenge): write an improvement plan
Done when three actions and three reasons are written, with data-related actions first.

In [ ]:
# 演習 3 / Exercise 3
plan = [
    {"priority": 1, "action": "", "why": ""},
    {"priority": 2, "action": "", "why": ""},
    {"priority": 3, "action": "", "why": ""},
]
for row in plan:
    print(f'{row["priority"]}. {row["action"] or "(未記入 / not written)"} '
          f'-> {row["why"] or "(未記入 / not written)"}')

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. precision と recall は、それぞれ何を測っていますか。
2. recall が低いと、現場でどんな問題が起きますか。
3. 混同行列の `background` の列が大きいとき、何が多いと言えますか。
4. 過学習を減らす最も効果的な方法は何ですか。
5. なぜ `split="test"` で評価するのですか。`split="train"` ではだめな理由も書いてください。

**English:**

1. What does each of precision and recall measure?
2. What problem does low recall cause on the line?
3. If the `background` column of the confusion matrix is large, what is frequent?
4. What is the most effective way to reduce overfitting?
5. Why evaluate with `split="test"`, and why is `split="train"` unacceptable?
#### English: Understanding questions
Model answers: `docs/instructor/day04_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `FileNotFoundError: 学習済みの重みが見つかりません` | `FileNotFoundError` about trained weights | 04_02 の学習を先に実行してください。`training_results/day04/` を確認してください。 | Run the training in 04_02 first; check `training_results/day04/`. |
| `model.val()` がエラーになる | `model.val()` raises an error | `data.yaml` の `path` が古い可能性があります。04_02 の Part 2 を再実行してください。 | `path` in `data.yaml` may be stale. Re-run Part 2 of 04_02. |
| 指標が 0 になる | All metrics are zero | 試験ラベルが空、またはクラス数が合っていない可能性があります。04_01 の点検に戻ってください。 | Test labels may be empty or the class count may differ. Repeat the Day 4 Notebook 01 checks. |
| 混同行列の画像が無い | No confusion matrix image | `plots=True` で学習していません。学習を再実行してください。 | Training was run without `plots=True`. Re-run it. |
| 共通画像セットが無い | The shared image set is missing | `python assets\scripts\prepare_datasets.py --only anomaly` を実行してください。 | Run `prepare_datasets.py --only anomaly`. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] `split="test"` で評価した
      - [ ] Evaluated with `split="test"`
- [ ] 混同行列を確認した
      - [ ] Inspected the confusion matrix
- [ ] 成功例と失敗例を画像で確認した
      - [ ] Inspected successes and failures visually
- [ ] `yolo_predictions_on_shared_set.json` を保存した
      - [ ] Saved `yolo_predictions_on_shared_set.json`
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics Val モード](https://docs.ultralytics.com/modes/val/)
- [混同行列の読み方](https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix)
- [precision と recall](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** `conf` を 0.1 と 0.6 にして `model.val()` を実行し、precision と recall の動きを比べてください。

**English:** Run `model.val()` with `conf` 0.1 and 0.6 and compare how precision and recall move.